# Phosphene Simulation Demo Pipeline

Hello! I'm Alicia and welcome to this project!

This notebook is a **minimal, portfolio-friendly demo** of the pipeline in this repository:

1. Load Dynaphos `params.yaml` and initialize the simulator  
2. Preprocess a sample image (grayscale / blur / Sobel / Canny)  
3. Render phosphenes for each preprocessing condition  
4. *(Optional)* Generate a short phosphene video from an input clip  

> **Privacy/Ethics note:** This notebook contains **no human-subject data** and no experimental results tables.


## 0) Setup

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on PYTHONPATH (so `import src...` works)
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

from src.simulator_setup import load_params, make_simulator
from src.preprocessing import PreprocessConfig, preprocess_frame
from src.phosphene_render import image_to_stimulus, to_uint8_image


## 1) Load params and initialize Dynaphos simulator

In [ ]:
# Path to your Dynaphos params.yaml
PARAMS_PATH = REPO_ROOT / "config" / "params.yaml"

params = load_params(PARAMS_PATH)
print("Loaded params keys:", list(params.keys())[:10])

# Choose number of phosphenes for this demo
N_PHOSPHENES = 1000

simulator, phosphene_coords = make_simulator(params, n_phosphenes=N_PHOSPHENES)
print("Simulator initialized with n_phosphenes =", N_PHOSPHENES)


## 2) Create a sample input image (synthetic, public-safe)

In [ ]:
# Create a simple synthetic image so the notebook runs without any external assets.
# You can replace this with your own public-domain image if you want.

H, W = 512, 512
img = np.zeros((H, W, 3), dtype=np.uint8)

# Draw a few shapes
cv2.circle(img, (160, 170), 80, (255, 255, 255), -1)
cv2.rectangle(img, (280, 120), (460, 300), (200, 200, 200), -1)
cv2.line(img, (60, 420), (460, 420), (255, 255, 255), 8)

plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Synthetic input image")
plt.axis("off")
plt.show()


## 3) Preprocess the image (gray / blur / Sobel / Canny)

In [ ]:
# Each mode returns a 256x256 uint8 image.
modes = ["none", "blur", "sobel", "canny"]
processed = {}

for mode in modes:
    cfg = PreprocessConfig(mode=mode, resolution=(256, 256))
    processed[mode] = preprocess_frame(img, cfg)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, mode in zip(axs, modes):
    ax.imshow(processed[mode], cmap="gray", origin="lower")
    ax.set_title(mode.upper())
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4) Render phosphenes for each preprocessing condition

In [ ]:
phosphene_imgs = {}

for mode in modes:
    stim = image_to_stimulus(simulator, processed[mode], rescale=True)
    simulator.reset()
    phs = simulator(stim)
    try:
        phs = phs.clamp(0, 1)
    except Exception:
        pass
    phosphene_imgs[mode] = to_uint8_image(phs)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, mode in zip(axs, modes):
    ax.imshow(phosphene_imgs[mode], cmap="gray", origin="lower")
    ax.set_title(f"PHOSPHENES: {mode.upper()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5) (Optional) Save a representative figure for the repo

In [ ]:
# Saves a single representative figure to results/media/
out_dir = REPO_ROOT / "results" / "media"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "phosphene_examples.png"

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, mode in zip(axs, modes):
    ax.imshow(phosphene_imgs[mode], cmap="gray", origin="lower")
    ax.set_title(mode.upper())
    ax.axis("off")
plt.tight_layout()
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_path)


## 6) (Optional) Generate a short phosphene video

In [ ]:
# Place a public-safe input clip at: assets/input.mp4 (or change the path below).

from src.video_io import generate_phosphene_video

INPUT_VIDEO = REPO_ROOT / "assets" / "input.mp4"
OUTPUT_VIDEO = REPO_ROOT / "results" / "media" / "example_phosphene_simulation.mp4"

if not INPUT_VIDEO.exists():
    print("Input video not found:", INPUT_VIDEO)
    print("Add a clip at assets/input.mp4 or update INPUT_VIDEO to your file.")
else:
    cfg = PreprocessConfig(mode="blur", resolution=(256, 256))
    generate_phosphene_video(
        input_video_path=INPUT_VIDEO,
        output_video_path=OUTPUT_VIDEO,
        simulator=simulator,
        preprocess_cfg=cfg,
        max_seconds=10.0,
        concat_processed_and_phosphenes=True,
    )
    print("Wrote:", OUTPUT_VIDEO)


---

## Done

For a public portfolio repo, keep it minimal:
- `src/` scripts
- this single notebook
- **one** image + **one** video in `results/media/`
- no participant data, no results tables
